In [170]:
import numpy as np
import pandas as pd
from glob import glob
from matplotlib import pyplot as plt
import seaborn as sns
import pickle as pkl
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from one.api import ONE
from sklearn.decomposition import PCA
from dPCA import dPCA
from sklearn.metrics import euclidean_distances
from scipy import stats
from manifold.pseudosession_manifolds import analyze_stitched_manifold

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import plotly.io as pio

pio.renderers.default = "notebook"

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
def get_region_stats():
    one = ONE()
    units_df = bwm_units(one)
    neuron_counts = units_df.groupby(["Beryl", "eid"]).size().reset_index(name="neuron_count")
    valid_pairings = neuron_counts[neuron_counts["neuron_count"] >= 5]
    # print(valid_pairings)  # atleast 10 neurons
    final_counts = (
        valid_pairings.groupby("Beryl")["eid"]
        .nunique()
        .reset_index(name="valid_eid_count")
        .sort_values(by="valid_eid_count", ascending=False)
    )
    # print(final_counts)
    regions_of_interest = final_counts[final_counts["valid_eid_count"] >= 20]["Beryl"].values
    region_totals = units_df.groupby("Beryl").size()
    valid_regions = region_totals[region_totals >= 20].index
    df_valid = units_df[units_df["Beryl"].isin(valid_regions)]

    final_table = (
        df_valid.groupby("Beryl")["eid"]
        .agg(
            total_neurons="size",
            unique_eid_count="nunique",
            eids_list=lambda x: list(x.unique()),
        )
        .reset_index()
        .sort_values(by="total_neurons", ascending=False)
    )
    # final_table[final_table["unique_eid_count"] >= 10]["Beryl"].values
    return final_table, final_counts

In [6]:
def plot_trajectories(traj_A, traj_B):
    import plotly.graph_objects as go

    trace_A = go.Scatter3d(
        x=traj_A[:, 0],
        y=traj_A[:, 1],
        z=traj_A[:, 2],
        mode="lines+markers",
        line=dict(color="blue", width=4),
        marker=dict(
            size=[8] + [2] * (len(traj_A) - 1), color="blue"
        ),  # Makes the first dot bigger
        name="Correct",
    )

    trace_B = go.Scatter3d(
        x=traj_B[:, 0],
        y=traj_B[:, 1],
        z=traj_B[:, 2],
        mode="lines+markers",
        line=dict(color="red", width=4),
        marker=dict(size=[8] + [2] * (len(traj_B) - 1), color="red"),  # Makes the first dot bigger
        name="Incorrect",
    )

    fig = go.Figure(data=[trace_A, trace_B])

    fig.update_layout(
        title="Trajectories in 3D Shared Subspace",
        scene=dict(xaxis_title="PC 1", yaxis_title="PC 2", zaxis_title="PC 3"),
        width=400,
        height=400,
    )
    fig.show()

In [87]:
def get_trajectory_data(data):

    stiched_session = []
    for k in data.keys():
        stiched_session.append(data[k])
    stiched_session = np.concatenate(stiched_session)

    pca = PCA(n_components=3)
    pca_session = pca.fit_transform(stiched_session)
    n_timepoints = 50
    cond_A_data = pca_session[:, 0:n_timepoints]
    cond_B_data = pca_session[:, n_timepoints:]

    plot_trajectories(cond_A_data, cond_B_data)

In [159]:
# def rearrange_distance_matrix(matrix):

#     final_structure = []
#     n_sessions = len(matrix[0][0])
#     n_eids = len(matrix)
#     for c in range(2):
#         session_layer = []
#         for s in range(n_sessions):

#             neurons_across_eids = [matrix[e][c][s] for e in range(n_eids)]
#             all_neurons_combined = np.concatenate(neurons_across_eids)
#             session_layer.append(all_neurons_combined)
#         final_structure.append(session_layer)

#     final_array = np.array(final_structure)
#     return final_array

In [162]:
files = np.sort(glob("../data/generated/manifold/true/*.pkl"))
pseudo_files = np.sort(glob("../data/generated/manifold/pseudo_metrics/*.pkl"))

In [ ]:
for idx in range(len(files)):
    fname = files[idx]
    fname_pseudo = pseudo_files[idx]
    with open(fname, "rb") as f:
        true_data = pkl.load(f)
    with open(fname_pseudo, "rb") as f:
        pseudo_data = pkl.load(f)
    true_x = analyze_stitched_manifold(true_data)
    rname = fname.rsplit(".pkl")[0].rsplit("_")[1]
    # single region check
    n_extreme = np.sum(pseudo_data["path_length_diff"] <= true_x["path_length_diff"])
    fig, ax = plt.subplots(figsize=(5, 5))
    sns.kdeplot(pseudo_data["path_length_diff"])
    ax.axvline(true_x["path_length_diff"])
    sns.despine()
    ax.set_title(f"Region: {rname}")

(np.float64(0.22529170168157442), np.float64(0.24513761630490657))